# 02 — Benchmark: 4-arm GraphRAG vs PlainRAG ablation

Thin Colab wrapper around `scripts/run_benchmark.py` and `scripts/compare.py`.

**Colab Pro:** Runtime → Change runtime type → **A100 GPU** + **High-RAM**. This run is
LLM-inference-bound (~800 generations from a reasoning model), so the faster GPU is what
cuts wall-clock. Add `ARANGO_PASS` in the Colab **Secrets** panel. Run `01_ingest.ipynb` first.

Arms (each isolates one component):

| arm | adds |
| --- | --- |
| `plain` | vector top-k chunks (baseline) |
| `plain_rr` | + cross-encoder reranker |
| `graph` | + parent-paper expansion (full abstracts) |
| `graph_concepts` | + MeSH concept-hop expansion |

The runner retries failed questions and auto-restarts Ollama if it crashes, and it
checkpoints every 25 questions — so a transient 500 can't abort an arm.

In [ ]:
# Confirm the GPU. A100 is ideal; T4/L4 also work (slower).
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/vardhjain/Knowledge_Graph_Question_Answering.git
%cd Knowledge_Graph_Question_Answering
!git checkout revamp
!pip install -q -r requirements.txt

In [ ]:
# Install Ollama and pull the LLM (once). The benchmark script manages the
# server from here on (health-check + auto-restart).
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version || true
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama pull deepseek-r1:8b

In [ ]:
import os
from google.colab import userdata
os.environ['ARANGO_PASS'] = userdata.get('ARANGO_PASS')
# On a small-VRAM GPU (e.g. T4) you can shrink generation if you hit OOM 500s:
# os.environ['LLM_NUM_CTX'] = '4096'      # default
# os.environ['LLM_NUM_PREDICT'] = '768'   # default 1024
print('ARANGO_PASS set:', bool(os.environ.get('ARANGO_PASS')))

In [ ]:
# Run all four arms. The chunk corpus is downloaded once and cached, then
# reused by every arm (identical corpus -> fair comparison). Each arm saves its
# own results JSON, so if one dies you can re-run just that arm.
for arm in ['plain', 'plain_rr', 'graph', 'graph_concepts']:
    print(f'\n===== {arm} =====')
    !python scripts/run_benchmark.py --arm {arm} --n 200

In [ ]:
# Summary table, paired McNemar tests, and the ablation figure.
!python scripts/compare.py
from IPython.display import Image, display, Markdown
display(Markdown(open('results/summary.md').read()))
display(Image('results/ablation.png'))

In [ ]:
# Optional: commit results back to GitHub (set a PAT first).
# !git config user.email you@example.com && git config user.name you
# !git add results/ && git commit -m 'Add benchmark results' && git push